In [1]:
import sys
sys.path.append('../../')  # go up two levels to root
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np
from utils.utils import nested_cv
from sklearn.multioutput import ClassifierChain


In [2]:
# load the data
X = pd.read_csv("../../data/processed/X_clinical.csv", sep=',')
y = pd.read_csv("../../data/processed/y.csv", sep=',')

# drop sampleId, patientId, and GENE_PANEL
X = X.drop(columns=['sampleId', 'patientId', 'GENE_PANEL'])
y = y.drop(columns=['sampleId', 'patientId'])

assert(len(X) == len(y))

# split data into train and testing set
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42)


In [3]:
print(y_train.sum(axis=0))  # how many positives per label in training set
print(y_test.sum(axis=0))   # how many positives per label in test set


[17 52 41 17 70 82 41]
[ 5 14 14  6 25 27 14]


In [4]:
from sklearn.dummy import DummyClassifier
dummy = MultiOutputClassifier(DummyClassifier(strategy='most_frequent'))
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)
print(f1_score(y_test, y_dummy, average='macro', zero_division=0))
print(accuracy_score(y_test, y_dummy))


0.0
0.4945054945054945


In [ ]:
# logisitic with l2 penalty
# C = 1/alpha
param_grid = {'estimator__C': np.logspace(-3, 3, 7)}

# balanced class weight adjusts the loss function so minority classes are penalized more heavily than the
model = MultiOutputClassifier(LogisticRegression(class_weight='balanced', max_iter=2000))
clf = GridSearchCV(estimator=model, param_grid=param_grid, 
                             cv=5, scoring='f1_macro') #searches for best params
clf.fit(X_train, y_train) # fit data on training fold
best_model= clf.best_estimator_            
y_pred = best_model.predict(X_test) # test 
print(f1_score(y_test, y_pred, average='macro', zero_division=0))
print(accuracy_score(y_test, y_pred))


0.34769173869783626
0.3626373626373626


In [6]:
y
# priority ordering will be defined as:
# CNS -> Bone -> Liver -> Adrenal -> Lung -> Pleura -> LN -> Non-metastatic


,EVER_MET_SITE_ADRENAL,EVER_MET_SITE_BONE,EVER_MET_SITE_CNS,EVER_MET_SITE_LIVER_BILIARY_TRACT,EVER_MET_SITE_LN,EVER_MET_SITE_LUNG,EVER_MET_SITE_PLEURA
0,0,1,0,0,1,0,0
1,0,1,0,1,0,0,0
2,1,1,1,0,1,0,1
3,0,0,0,0,0,1,0
4,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...
450,0,0,0,0,0,1,0
451,0,0,0,1,1,1,0
452,0,0,1,0,0,0,1
453,1,0,0,0,0,0,0


In [12]:
# logisitic with l2 penalty
# C = 1/alpha
param_grid = {'estimator__C': np.logspace(-3, 3, 7)}

order = [2,1,3,0,5,6,4]
# order = [0,1,2,3,4,5,6]
# balanced class weight adjusts the loss function so minority classes are penalized more heavily than the majority class.
model = ClassifierChain(LogisticRegression(class_weight='balanced', max_iter=2000), order=order)
clf = GridSearchCV(estimator=model, param_grid=param_grid, 
                             cv=5, scoring='f1_macro') #searches for best params
clf.fit(X_train, y_train) # fit data on training fold
best_model= clf.best_estimator_            
y_pred = best_model.predict(X_test) # test 
print(f1_score(y_test, y_pred, average='macro', zero_division=0))
print(accuracy_score(y_test, y_pred))


0.3480357424497256
0.38461538461538464


In [ ]:
model_no_penalty = ClassifierChain(LogisticRegression(penalty=None, class_weight='balanced', max_iter=2000),  order=order)
clf = GridSearchCV(estimator=model, param_grid=param_grid, 
                             cv=5, scoring='f1_macro') #searches for best params
clf.fit(X_train, y_train) # fit data on training fold
best_model= clf.best_estimator_            
y_pred = best_model.predict(X_test) # test 
print(f1_score(y_test, y_pred, average='macro', zero_division=0))
print(accuracy_score(y_test, y_pred))


0.3480357424497256
0.38461538461538464


In [9]:
nested_scores= nested_cv(model=model, p_grid=param_grid, X=X_train, y=y_train)


In [10]:
nested_scores


array([0.28643276, 0.29053176, 0.29013987, 0.27063311, 0.29973105,
       0.29403768, 0.30213845, 0.30181869, 0.29303863, 0.2692878 ])